<a href="https://colab.research.google.com/github/romavallejo/TC3009C.600_AIClass/blob/main/P1_M1_%20PythonLib_Statistics/meteorolog%C3%ADa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd

#Extraction
raw_df = pd.read_csv('meteorología_2025.csv', sep=',', skiprows=9)

display(raw_df.head())

,date,id_station,id_parameter,valor,unit
0,2025-01-01 00:00:00,ACO,TMP,8.7,5
1,2025-01-01 00:00:00,ACO,RH,28.0,6
2,2025-01-01 00:00:00,ACO,WSP,2.4,3
3,2025-01-01 00:00:00,ACO,WDR,2.0,4
4,2025-01-01 00:00:00,AJM,TMP,9.6,5


Humedad == RH
viento == WSP
---
humedad alta >= 70
viento tranquilo <= 1

In [25]:
#Transformation
# columna date a tipo datetime
df = raw_df.copy()
df['date'] = pd.to_datetime(df['date'])

In [26]:
# revisar información en las estaciones
total_por_estacion = df.groupby('id_station').size()
nulos_por_estacion = df[df['valor'].isnull()].groupby('id_station').size()
pct_nulo = (nulos_por_estacion / total_por_estacion * 100).sort_values(ascending=False)
print(pct_nulo)

id_station
CAM    100.000000
ATI    100.000000
SFE    100.000000
TLA    100.000000
CCA    100.000000
HGM    100.000000
IZT    100.000000
LPR    100.000000
TLI    100.000000
XAL    100.000000
AJU     91.332763
SAC     48.781393
ACO     43.013699
CUA     35.958904
FAR     35.956050
BJU     33.672945
CHO     33.518836
TAH     33.467466
MON     32.948059
MPA     29.118151
FAC     28.655822
SAG     26.857877
LAA     25.625000
VIF     25.496575
INN     23.598744
UIZ     19.783105
GAM     19.660388
NEZ     15.268265
PED      0.847603
MGH      0.322489
AJM      0.305365
CUT      0.273973
MER      0.199772
UAX      0.034247
dtype: float64


In [27]:
# Limpieza final
umbral_exclusion = 90  # estaciones con >=90% de nulos se excluyen

df_util = df[~df['id_station'].isin(pct_nulo[pct_nulo >= umbral_exclusion].index)].copy()
df_clean = df_util.dropna(subset=['valor']).copy()

print(f"Filas: {len(df)} → {len(df_util)} (sin estaciones inútiles) → {len(df_clean)} (sin nulos)")

# Usa df_clean (ya sin nulos) filtrado a GAM
df_gam_clean = df_clean[df_clean['id_station'] == 'GAM']

# pivotear para tener RH y WSP como columnas alineadas por fecha
df_gam_wide = df_gam_clean.pivot_table(index='date', columns='id_parameter', values='valor')

# filas donde tenemos ambas mediciones ese mismo momento
df_gam_wide = df_gam_wide.dropna(subset=['RH', 'WSP'])

print(f"Observaciones con ambas mediciones: {len(df_gam_wide)}")

Filas: 1191360 → 805920 (sin estaciones inútiles) → 626037 (sin nulos)
Observaciones con ambas mediciones: 6900


In [28]:
#seleccionamos la gustavo a madero para el ejercicio
df_gam = df_clean[df_clean['id_station'] == 'GAM'].copy()
df_gam.head()

,date,id_station,id_parameter,valor,unit
16368,2025-01-06 00:00:00,GAM,TMP,12.9,5
16369,2025-01-06 00:00:00,GAM,RH,74.0,6
16504,2025-01-06 01:00:00,GAM,TMP,12.5,5
16505,2025-01-06 01:00:00,GAM,RH,79.0,6
16640,2025-01-06 02:00:00,GAM,TMP,12.0,5


In [29]:
# Marginales
p_humedad_alta = (df_gam_wide['RH'] >= 70).mean()
p_viento_tranquilo = (df_gam_wide['WSP'] <= 1).mean()

# Conjunta: ambas condiciones en la MISMA fila (mismo instante)
p_ambas = ((df_gam_wide['RH'] >= 70) & (df_gam_wide['WSP'] <= 1)).mean()

print(f"P(humedad alta) = {p_humedad_alta*100:.2f}%")
print(f"P(viento tranquilo) = {p_viento_tranquilo*100:.2f}%")
print(f"P(humedad alta y viento tranquilo) = {p_ambas*100:.2f}%")

# Condicionales correctas: P(A|B) = P(A y B) / P(B)
p_humedad_dado_viento = p_ambas / p_viento_tranquilo
p_viento_dado_humedad = p_ambas / p_humedad_alta

print(f"P(humedad alta | viento tranquilo) = {p_humedad_dado_viento*100:.2f}%")
print(f"P(viento tranquilo | humedad alta) = {p_viento_dado_humedad*100:.2f}%")

P(humedad alta) = 32.97%
P(viento tranquilo) = 28.81%
P(humedad alta y viento tranquilo) = 17.90%
P(humedad alta | viento tranquilo) = 62.12%
P(viento tranquilo | humedad alta) = 54.29%
